In [2]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "scripts")
print(os.getcwd())

b:\TrailGuard-1\trailguard\tests


In [3]:
import json
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score, roc_auc_score
from builder_lanl_b2 import build_from_files, WINDOW_SECONDS
from unsupervised_helper_b2 import train_unsupervised, get_normalized_unsup_score

In [4]:
USE_TOY = False

if USE_TOY:
    AUTH, RED = "data/raw/toy_subset.csv.gz", "data/raw/toy_redteam_hashed.csv.gz"
    OUT_CSV, OUT_JSON = "data/processed/toy_features.csv.gz", "data/processed/toy_columns.json"
else:
    AUTH, RED = "data/raw/lanl_subset_trim.csv.gz", "data/raw/lanl_redteam_hashed.csv.gz"
    OUT_CSV, OUT_JSON = "data/processed/features_lanl.csv.gz", "data/processed/lanl_feature_columns.json"

In [5]:
feat = build_from_files(AUTH, RED, OUT_CSV, OUT_JSON)
print(feat.shape, "| positives:", int(feat.label.sum()), "| rate: %.4f" % feat.label.mean())
feat.head()

(158679, 15) | positives: 282 | rate: 0.0018


,time_key,user_hash,label,total_auth_events,fail_auth_count,fail_auth_ratio,off_hours_event_ratio,unique_src_computers,unique_dst_computers,new_dst_computer_count,network_logon_ratio,kerberos_auth_ratio,total_auth_events_z,unique_dst_computers_z,fail_auth_count_z
0,0,05fa1d199c79e66b,0,120.0,0.0,0.0,1.0,15.0,14.0,14.0,0.783333,0.358333,0.0,0.0,0.0
1,0,064933cf519d015f,0,10.0,0.0,0.0,1.0,1.0,1.0,1.0,0.500000,0.000000,0.0,0.0,0.0
2,0,099c04a9b2b8e8a0,0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.000000,0.000000,0.0,0.0,0.0
3,0,141a3e4dc6e70b0f,0,7178.0,0.0,0.0,1.0,101.0,104.0,104.0,0.968097,0.449707,0.0,0.0,0.0
4,0,207723d77fff7903,0,64.0,0.0,0.0,1.0,9.0,6.0,6.0,0.906250,0.390625,0.0,0.0,0.0


In [6]:
def temporal_split(df):
    df = df.sort_values("time_key", kind="stable").reset_index(drop=True)
    n = len(df); a, b = int(n * 0.70), int(n * 0.85)
    return df.iloc[:a], df.iloc[a:b], df.iloc[b:]

def prepare(df):
    tr, va, te = temporal_split(df)
    cols = [c for c in df.columns if c not in ("time_key", "user_hash", "label")]
    imp = SimpleImputer(strategy="median").fit(tr[cols])          # fit on TRAIN only
    sc = RobustScaler().fit(imp.transform(tr[cols]))
    f = lambda d: sc.transform(imp.transform(d[cols]))
    out = dict(cols=cols, imp=imp, sc=sc,
               Xtr=f(tr), Xva=f(va), Xte=f(te),
               ytr=tr.label.values, yva=va.label.values, yte=te.label.values,
               tk_va=va.time_key.values, tk_te=te.time_key.values)
    for name, d in [("train", tr), ("val", va), ("test", te)]:
        print(f"{name:5s} rows={len(d):8d} positives={int(d.label.sum()):5d}")
    return out

def metrics(y, s):
    if y.sum() == 0:
        return dict(pr_auc=np.nan, roc_auc=np.nan)
    return dict(pr_auc=average_precision_score(y, s), roc_auc=roc_auc_score(y, s))

def budget_metrics(time_key, y, score, ks=(5, 10, 20)):
    day = (time_key * WINDOW_SECONDS) // 86400
    d = pd.DataFrame(dict(day=day, y=y, s=score))
    res = {}
    for k in ks:
        sel = d.sort_values("s", ascending=False).groupby("day").head(k)
        res[f"P@{k}/day"] = round(sel.y.sum() / max(len(sel), 1), 3)
        res[f"R@{k}/day"] = round(sel.y.sum() / max(d.y.sum(), 1), 3)
    return res

In [7]:
D = prepare(feat)
if D["ytr"].sum() == 0:
    print("NOTE: no attacks in Train. Known limitation when red-team activity is clustered in time.")

train rows=  111075 positives=  282
val   rows=   23802 positives=    0
test  rows=   23802 positives=    0


In [8]:
pos = feat[feat.label == 1]
cut70 = feat.sort_values("time_key").iloc[int(len(feat) * 0.70)].time_key
cut85 = feat.sort_values("time_key").iloc[int(len(feat) * 0.85)].time_key
print("overall time_key range :", feat.time_key.min(), "to", feat.time_key.max())
print("attack  time_key range :", pos.time_key.min(), "to", pos.time_key.max())
print("train ends at time_key :", cut70, "| val ends at:", cut85)
print("attack days            :", pos.time_key.min() * WINDOW_SECONDS // 86400, "to", pos.time_key.max() * WINDOW_SECONDS // 86400)
print("total days             :", feat.time_key.max() * WINDOW_SECONDS // 86400)

overall time_key range : 0 to 1391
attack  time_key range : 41 to 710
train ends at time_key : 988 | val ends at: 1187
attack days            : 1 to 29
total days             : 57


In [9]:
# Extra evaluation: split in the middle of the attack period (still strictly time-ordered)
feat_s = feat.sort_values("time_key", kind="stable").reset_index(drop=True)
split_tk = int(feat_s[feat_s.label == 1].time_key.median())
tr = feat_s[feat_s.time_key <= split_tk]
te = feat_s[feat_s.time_key > split_tk]
cols = [c for c in feat_s.columns if c not in ("time_key", "user_hash", "label")]

imp = SimpleImputer(strategy="median").fit(tr[cols])           # fit on train only
sc = RobustScaler().fit(imp.transform(tr[cols]))
Xtr2 = sc.transform(imp.transform(tr[cols]))
Xte2 = sc.transform(imp.transform(te[cols]))
print("split at time_key", split_tk, "| train rows", len(tr), "attacks", int(tr.label.sum()),
      "| test rows", len(te), "attacks", int(te.label.sum()))

model2, a2, b2 = train_unsupervised(Xtr2, tr.label.values)      # fits on normal rows only
assert 0.0 <= get_normalized_unsup_score(model2, Xte2, a2, b2).max() <= 1.0
s_iso = -model2.score_samples(Xte2)                              # raw score, avoids ties
s_z = np.abs(Xte2).max(axis=1)

print("IsolationForest :", metrics(te.label.values, s_iso))
print("Max |Z| baseline:", metrics(te.label.values, s_z))
print("IsolationForest alert budget:", budget_metrics(te.time_key.values, te.label.values, s_iso))

split at time_key 299 | train rows 32925 attacks 150 | test rows 125754 attacks 132
IsolationForest : {'pr_auc': 0.021879257617824718, 'roc_auc': 0.8044761388542733}
Max |Z| baseline: {'pr_auc': 0.005099486883743578, 'roc_auc': 0.7399813377120299}
IsolationForest alert budget: {'P@5/day': np.float64(0.035), 'R@5/day': np.float64(0.061), 'P@10/day': np.float64(0.022), 'R@10/day': np.float64(0.076), 'P@20/day': np.float64(0.013), 'R@20/day': np.float64(0.091)}


In [7]:
results = []
for n_est in (100, 200, 300):
    for ms in ("auto", 256, 1024):
        m = IsolationForest(n_estimators=n_est, max_samples=ms, contamination="auto",
                            random_state=42, n_jobs=-1)
        m.fit(D["Xtr"][D["ytr"] == 0])
        s = -m.score_samples(D["Xva"])
        results.append(dict(model="IsolationForest", n_estimators=n_est, max_samples=ms,
                            **metrics(D["yva"], s)))

lof = LocalOutlierFactor(n_neighbors=35, novelty=True).fit(D["Xtr"][D["ytr"] == 0])
results.append(dict(model="LOF(novelty)", n_estimators=None, max_samples=None,
                    **metrics(D["yva"], -lof.score_samples(D["Xva"]))))

zmax = np.abs(D["Xva"]).max(axis=1)
results.append(dict(model="Max |Z| baseline", n_estimators=None, max_samples=None,
                    **metrics(D["yva"], zmax)))

res = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
res

,model,n_estimators,max_samples,pr_auc,roc_auc
8,IsolationForest,300.0,1024,1.000000,1.000000
1,IsolationForest,100.0,256,1.000000,1.000000
0,IsolationForest,100.0,auto,1.000000,1.000000
2,IsolationForest,100.0,1024,1.000000,1.000000
3,IsolationForest,200.0,auto,1.000000,1.000000
5,IsolationForest,200.0,1024,1.000000,1.000000
4,IsolationForest,200.0,256,1.000000,1.000000
6,IsolationForest,300.0,auto,1.000000,1.000000
7,IsolationForest,300.0,256,1.000000,1.000000
10,Max |Z| baseline,NaN,None,0.653169,0.982767


In [ ]:
best = res[res.model == "IsolationForest"].iloc[0]
ms = best.max_samples if best.max_samples == "auto" else int(best.max_samples)
model, min_s, max_s = train_unsupervised(D["Xtr"], D["ytr"], n_estimators=int(best.n_estimators), max_samples=ms)
print("chosen:", int(best.n_estimators), ms, "| min_s=%.4f max_s=%.4f" % (min_s, max_s))

test_score = get_normalized_unsup_score(model, D["Xte"], min_s, max_s)
assert 0.0 <= test_score.min() and test_score.max() <= 1.0

print("TEST:", metrics(D["yte"], test_score))
print("TEST alert budget:", budget_metrics(D["tk_te"], D["yte"], test_score))
print("Positives are very sparse, so treat these numbers as indicative, not conclusive.")

chosen: 300 1024 | min_s=0.3453 max_s=0.6467
TEST: {'pr_auc': 0.37254901960784315, 'roc_auc': 0.994417306350314}
TEST alert budget: {'P@5/day': np.float64(0.4), 'R@5/day': np.float64(0.526), 'P@10/day': np.float64(0.38), 'R@10/day': np.float64(1.0), 'P@20/day': np.float64(0.19), 'R@20/day': np.float64(1.0)}
Positives are very sparse, so treat these numbers as indicative, not conclusive.


In [10]:
print(USE_TOY)
print(AUTH)

True
data/raw/toy_subset.csv.gz
